In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

# ---- Configuration ----
L = 32
BETAS = [3.6, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]
N_STEPS = 600
N_MEASURE = 50
BATCH_SIZE = 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Simulation Engine Active: {torch.cuda.get_device_name(0)}")

# ---- Physics Kernels ----

def generate_updates(shape, epsilon):
    """Generates random SU(3) matrices close to Identity using Matrix Exp"""
    # Random anti-hermitian generator
    R = torch.randn(*shape, dtype=torch.cfloat, device=device)
    A = 0.5 * (R - torch.conj(R.transpose(-2, -1)))

    # Make traceless
    tr = torch.einsum('...ii->...', A)
    I = torch.eye(3, device=device, dtype=torch.cfloat)
    A = A - (tr.unsqueeze(-1).unsqueeze(-1) / 3.0) * I

    # Exact exponentiation for SU(3) compliance
    return torch.linalg.matrix_exp(epsilon * A)

def compute_staples(U, mu):
    """
    Computes the sum of the 6 'staples' surrounding links in direction mu.
    Result has shape (Batch, L, L, L, L, 3, 3) matching U[:, mu].
    """
    staple_sum = torch.zeros_like(U[:, mu])

    for nu in range(4):
        if nu == mu: continue

        # --- Staple 'Up' (in direction nu) ---
        # U_nu(x) * U_mu(x+nu) * U_nu(x+mu)^dag
        # Shift definitions:
        # U_nu(x)       -> U[:, nu]
        # U_mu(x+nu)    -> roll U[:, mu] shift -1 in dim nu+1
        # U_nu(x+mu)    -> roll U[:, nu] shift -1 in dim mu+1

        U_nu = U[:, nu]
        U_mu_shift_nu = torch.roll(U[:, mu], shifts=-1, dims=nu+1)
        U_nu_shift_mu = torch.roll(U[:, nu], shifts=-1, dims=mu+1)

        up = torch.matmul(U_nu, U_mu_shift_nu)
        up = torch.matmul(up, U_nu_shift_mu.transpose(-2, -1).conj())
        staple_sum += up

        # --- Staple 'Down' (in direction -nu) ---
        # U_nu(x-nu)^dag * U_mu(x-nu) * U_nu(x+mu-nu)

        U_nu_down = torch.roll(U[:, nu], shifts=1, dims=nu+1)
        U_mu_down = torch.roll(U[:, mu], shifts=1, dims=nu+1)
        U_nu_shift_mu_down = torch.roll(U[:, nu], shifts=1, dims=nu+1) # First shift down
        U_nu_shift_mu_down = torch.roll(U_nu_shift_mu_down, shifts=-1, dims=mu+1) # Then shift mu

        down = torch.matmul(U_nu_down.transpose(-2, -1).conj(), U_mu_down)
        down = torch.matmul(down, U_nu_shift_mu_down)
        staple_sum += down

    return staple_sum

def measure_roughness(U):
    """Measures deviation from identity (approx A_mu field norm)"""
    # Project to anti-hermitian part (approx i*A)
    Ah = 0.5 * (U - torch.conj(U.transpose(-2, -1)))
    # Traceless
    tr = torch.einsum('b...ii->b...', Ah)
    I = torch.eye(3, device=device, dtype=torch.cfloat)
    Ah = Ah - (tr.unsqueeze(-1).unsqueeze(-1) / 3.0) * I

    # Norm calculation
    norm2 = torch.real(torch.einsum('b...ij,b...ji->b...', torch.conj(Ah), Ah))
    return torch.sqrt(norm2).view(BATCH_SIZE, -1).mean(dim=1)

# ---- Simulation Loop ----
print(f"\n--- Starting SU(3) Simulation (L={L}) ---")
results = []

# Checkerboard Masks (Pre-computed)
# Creates a boolean mask where (x+y+z+t) is even or odd
coords = torch.stack(torch.meshgrid(
    torch.arange(L), torch.arange(L), torch.arange(L), torch.arange(L), indexing='ij'
), dim=-1).to(device)
parity_sum = coords.sum(dim=-1)
mask_even = (parity_sum % 2 == 0)
mask_odd = (parity_sum % 2 == 1)

for beta in BETAS:
    print(f"Sampling Beta = {beta:<4.1f} ... ", end="")
    t0 = time.time()

    # Cold Start (Identity) or Hot Start (Random)
    # Using Cold Start to thermalize safer at high beta, or Hot for general
    U = generate_updates((BATCH_SIZE, 4, L, L, L, L, 3, 3), epsilon=10.0)
    # Re-project to ensure perfect unitarity after random init
    U, _ = torch.linalg.qr(U)

    measure_history = []

    # Adaptive epsilon logic
    epsilon = 0.2 if beta < 5.0 else 0.1

    for step in range(N_STEPS + N_MEASURE):

        # Loop over Directions (Mu)
        for mu in range(4):
            # Calculate Staples (Interaction with neighbors)
            staples = compute_staples(U, mu)

            # Loop over Parity (Checkerboard)
            for parity_mask in [mask_even, mask_odd]:

                # Get current links and staples for this parity
                # Note: We keep dimensions to allow broadcasting
                link_old = U[:, mu]
                staple_curr = staples

                # Generate Updates
                X = generate_updates(link_old.shape, epsilon)
                link_new = torch.matmul(X, link_old)

                # Calculate Local Action Change
                # dS = - (beta/3) * ReTr( (U_new - U_old) * Staple_dag )
                # Note: Maximize ReTr(U*S_dag) corresponds to minimizing Action

                prod_old = torch.matmul(link_old, staple_curr.transpose(-2, -1).conj())
                prod_new = torch.matmul(link_new, staple_curr.transpose(-2, -1).conj())

                S_old_local = torch.real(torch.einsum('b...ii->b...', prod_old))
                S_new_local = torch.real(torch.einsum('b...ii->b...', prod_new))

                dS = -(beta / 3.0) * (S_new_local - S_old_local) # Standard Wilson Action sign

                # Metropolis Test
                # If dS < 0 (Action decreases), exp(-dS) > 1 -> Accept
                # If dS > 0 (Action increases), Accept with prob exp(-dS)
                # Correction: The term calculated above is actually d(-Action) because we used Trace.
                # Let's simplify: We want to MAXIMIZE Trace.
                # d_Trace = S_new_local - S_old_local
                # Probability ~ exp( beta/3 * d_Trace )

                d_Trace = S_new_local - S_old_local
                prob = torch.exp( (beta/3.0) * d_Trace )
                rand = torch.rand_like(prob)

                accept = rand < prob

                # Apply mask: Only update if Accept AND Parity matches
                final_mask = accept & parity_mask.unsqueeze(0) # Broadcast batch

                # Update U
                final_mask_expanded = final_mask.view(*final_mask.shape, 1, 1)
                U[:, mu] = torch.where(final_mask_expanded, link_new, link_old)

        # Measurement
        if step >= N_STEPS:
            r = measure_roughness(U)
            measure_history.append(r.mean().item())

    # Stats
    r_avg = np.mean(measure_history)
    dt = time.time() - t0
    print(f"Done ({dt:.1f}s) | Field Roughness = {r_avg:.6f}")
    results.append((beta, r_avg))

# ---- Plot Results ----

betas_np = np.array([b for b, r in results])
rs_np = np.array([r for b, r in results])

plt.figure(figsize=(8,6))
plt.plot(betas_np, rs_np, 'o-', color='cyan', lw=2, label='Simulation')

# Theoretical scaling: A ~ 1/sqrt(beta) for non-Abelian fields in 4D
# (Naive dimensional analysis for vector potentials)
ref_scaling = rs_np[0] * np.sqrt(betas_np[0] / betas_np)
plt.plot(betas_np, ref_scaling, 'w--', alpha=0.5, label=r'$\sim 1/\sqrt{\beta}$')

plt.title(f"SU(3) Field Roughness Scaling (L={L})", color='white')
plt.xlabel("Beta", color='white')
plt.ylabel("Average ||A_mu||", color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor('#1e1e1e')
plt.gcf().patch.set_facecolor('#1e1e1e')
plt.tick_params(colors='white')
plt.show()

print("\n# ---- Copy this block into YANG3 ----")
print("R_curve = [")
for b, r in results:
    print(f"  ({b:.1f}, {r:.8f}),")
print("]")

AssertionError: Torch not compiled with CUDA enabled